# U-Net++ 3-fold talc segmentation

3-fold cross-validation for manually corrected talc masks. Default model is `UnetPlusPlus` with `efficientnet-b3`, because this encoder was more stable on the first runs.

For ConvNeXt Large in SMP use the timm wrapper name: `tu-convnext_large.fb_in22k_ft_in1k_384`. In SMP 0.5 it does not work with `UnetPlusPlus` because the timm encoder exposes a zero-channel early skip. Use `Unet`, `FPN`, or `MAnet` for that encoder.


In [ ]:
# If needed, run this once on the server kernel.
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# !pip install segmentation-models-pytorch albumentations wandb torchmetrics timm scikit-learn


In [ ]:
from pathlib import Path
from datetime import datetime
import os
import random
import subprocess

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import KFold, StratifiedKFold
import segmentation_models_pytorch as smp
import wandb

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('smp:', smp.__version__)


In [ ]:
SEED = 117
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DATA_ROOT = Path('/home/team117/nornik/talc_unetpp_dataset_from_json')
RUNS_DIR = Path('/home/team117/nornik/runs_unetpp_3fold')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    'project': 'nornik-talc-segmentation',
    'run_name': 'unetpp-efficientnet-b3-512-3fold',
    'group': 'unetpp-efficientnet-b3-512-3fold',
    'data_root': str(DATA_ROOT),
    'model': 'UnetPlusPlus',
    'encoder': 'efficientnet-b3',
    'encoder_weights': 'imagenet',
    'img_size': 512,
    'batch_size': 8,
    'epochs': 35,
    'lr': 2e-4,
    'weight_decay': 1e-4,
    'num_workers': 4,
    'n_folds': 3,
    'folds_to_run': [0, 1, 2],
    'dice_weight': 1.0,
    'bce_weight': 1.0,
    'threshold': 0.5,
    'early_stopping_patience': 8,
    'early_stopping_min_delta': 1e-4,
    'log_gpu_stats': True,
    'seed': SEED,
}

# ConvNeXt Large trial preset. It is heavy; start with img_size=384 and batch_size=2..4.
# Important: this timm ConvNeXt encoder is not compatible with SMP UnetPlusPlus,
# because its early skip has 0 channels. Use Unet, FPN, or MAnet.
# CFG.update({
#     'run_name': 'unet-convnext-large-384-3fold',
#     'group': 'unet-convnext-large-384-3fold',
#     'model': 'Unet',
#     'encoder': 'tu-convnext_large.fb_in22k_ft_in1k_384',
#     'encoder_weights': 'imagenet',
#     'img_size': 384,
#     'batch_size': 2,
#     'lr': 1e-4,
# })

assert (DATA_ROOT / 'images').exists(), DATA_ROOT
assert (DATA_ROOT / 'masks').exists(), DATA_ROOT

run_id = datetime.now().strftime('%Y%m%d-%H%M%S')
BASE_OUT_DIR = RUNS_DIR / f"{run_id}_{CFG['run_name']}"
BASE_OUT_DIR.mkdir(parents=True, exist_ok=True)
print(BASE_OUT_DIR)


In [ ]:
def list_all_pairs():
    img_dir = DATA_ROOT / 'images'
    mask_dir = DATA_ROOT / 'masks'
    image_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}])
    pairs = []
    for img_path in image_paths:
        matches = sorted(mask_dir.glob(img_path.stem + '.*'))
        if not matches:
            raise FileNotFoundError(f'No mask for {img_path.name}')
        pairs.append((img_path, matches[0]))
    return pairs

def mask_fraction(mask_path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(mask_path)
    return float((mask > 127).mean())

pairs = list_all_pairs()
meta = pd.DataFrame({
    'image': [p[0].name for p in pairs],
    'mask': [p[1].name for p in pairs],
    'mask_fraction': [mask_fraction(p[1]) for p in pairs],
})

try:
    meta['strata'] = pd.qcut(meta['mask_fraction'], q=CFG['n_folds'], labels=False, duplicates='drop')
    splitter = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=SEED)
    split_iter = splitter.split(meta, meta['strata'])
except ValueError:
    meta['strata'] = 0
    splitter = KFold(n_splits=CFG['n_folds'], shuffle=True, random_state=SEED)
    split_iter = splitter.split(meta)

fold_indices = list(split_iter)
meta['fold'] = -1
for fold, (_, val_idx) in enumerate(fold_indices):
    meta.loc[val_idx, 'fold'] = fold

meta.to_csv(BASE_OUT_DIR / 'folds.csv', index=False)
print(meta.groupby('fold').agg(n=('image', 'count'), mask_fraction_mean=('mask_fraction', 'mean'), mask_fraction_min=('mask_fraction', 'min'), mask_fraction_max=('mask_fraction', 'max')))
meta.head()


In [ ]:
class TalcSegDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype('float32')

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            mask = torch.from_numpy(mask).float()

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        return image, mask, img_path.name

train_tfms = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=20, border_mode=cv2.BORDER_REFLECT_101, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.6),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=10, p=0.35),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.25),
    A.Blur(blur_limit=3, p=0.15),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


In [ ]:
def denormalize(x):
    mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(3, 1, 1)
    return torch.clamp(x * std + mean, 0, 1)

def dice_loss_from_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dims)
    union = torch.sum(probs, dims) + torch.sum(targets, dims)
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()

def batch_metrics(logits, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    dims = (1, 2, 3)
    inter = torch.sum(preds * targets, dims)
    pred_sum = torch.sum(preds, dims)
    target_sum = torch.sum(targets, dims)
    union = pred_sum + target_sum - inter
    dice = (2 * inter + eps) / (pred_sum + target_sum + eps)
    iou = (inter + eps) / (union + eps)
    return {'dice': dice.mean().item(), 'iou': iou.mean().item()}

def make_wandb_images(images, masks, logits, names, max_items=4):
    images = images.detach().cpu()
    masks = masks.detach().cpu().numpy()
    probs = torch.sigmoid(logits.detach()).cpu().numpy()
    out = []
    for i in range(min(max_items, images.shape[0])):
        img = denormalize(images[i]).permute(1, 2, 0).numpy()
        gt = (masks[i, 0] > 0.5).astype(np.uint8)
        pred = (probs[i, 0] > CFG['threshold']).astype(np.uint8)
        out.append(wandb.Image(
            (img * 255).astype(np.uint8),
            caption=names[i],
            masks={
                'ground_truth': {'mask_data': gt, 'class_labels': {0: 'background', 1: 'talc'}},
                'prediction': {'mask_data': pred, 'class_labels': {0: 'background', 1: 'talc'}},
            }
        ))
    return out

def get_gpu_stats(device):
    stats = {}
    if device.type != 'cuda' or not CFG.get('log_gpu_stats', True):
        return stats
    stats.update({
        'gpu/torch_mem_allocated_gb': torch.cuda.memory_allocated() / 1024**3,
        'gpu/torch_mem_reserved_gb': torch.cuda.memory_reserved() / 1024**3,
        'gpu/torch_max_mem_allocated_gb': torch.cuda.max_memory_allocated() / 1024**3,
    })
    try:
        query = 'utilization.gpu,utilization.memory,memory.used,memory.total,power.draw,temperature.gpu'
        out = subprocess.check_output([
            'nvidia-smi', f'--query-gpu={query}', '--format=csv,noheader,nounits'
        ], text=True, timeout=5)
        values = [float(x.strip()) for x in out.strip().splitlines()[0].split(',')]
        stats.update({
            'gpu/utilization_pct': values[0],
            'gpu/memory_utilization_pct': values[1],
            'gpu/memory_used_mb': values[2],
            'gpu/memory_total_mb': values[3],
            'gpu/power_w': values[4],
            'gpu/temperature_c': values[5],
        })
    except Exception as exc:
        stats['gpu/nvidia_smi_error'] = str(exc)[:200]
    return stats


In [ ]:
def build_model(device):
    model_cls = getattr(smp, CFG['model'])
    model = model_cls(
        encoder_name=CFG['encoder'],
        encoder_weights=CFG['encoder_weights'],
        in_channels=3,
        classes=1,
        activation=None,
    )
    return model.to(device)

def make_loaders(train_pairs, val_pairs):
    train_ds = TalcSegDataset(train_pairs, train_tfms)
    val_ds = TalcSegDataset(val_pairs, val_tfms)
    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'], pin_memory=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'], pin_memory=True, drop_last=False)
    return train_loader, val_loader

def train_one_epoch(model, loader, optimizer, scaler, total_loss, device):
    model.train()
    losses, dices, ious = [], [], []
    for images, masks, names in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(images)
            loss = total_loss(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        m = batch_metrics(logits, masks, CFG['threshold'])
        losses.append(loss.item()); dices.append(m['dice']); ious.append(m['iou'])
    return {'loss': float(np.mean(losses)), 'dice': float(np.mean(dices)), 'iou': float(np.mean(ious))}

@torch.no_grad()
def validate(model, loader, total_loss, device, log_images=False):
    model.eval()
    losses, dices, ious = [], [], []
    preview = None
    for images, masks, names in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).float()
        with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(images)
            loss = total_loss(logits, masks)
        m = batch_metrics(logits, masks, CFG['threshold'])
        losses.append(loss.item()); dices.append(m['dice']); ious.append(m['iou'])
        if log_images and preview is None:
            preview = make_wandb_images(images, masks, logits, names)
    metrics = {'loss': float(np.mean(losses)), 'dice': float(np.mean(dices)), 'iou': float(np.mean(ious))}
    return metrics, preview


In [ ]:
def run_fold(fold):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fold_out = BASE_OUT_DIR / f'fold_{fold}'
    fold_out.mkdir(parents=True, exist_ok=True)

    train_idx, val_idx = fold_indices[fold]
    train_pairs = [pairs[i] for i in train_idx]
    val_pairs = [pairs[i] for i in val_idx]
    train_loader, val_loader = make_loaders(train_pairs, val_pairs)

    model = build_model(device)
    bce_loss = nn.BCEWithLogitsLoss()

    def total_loss(logits, targets):
        return CFG['bce_weight'] * bce_loss(logits, targets) + CFG['dice_weight'] * dice_loss_from_logits(logits, targets)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=CFG['lr'] * 0.05)
    scaler = GradScaler(device.type, enabled=(device.type == 'cuda'))

    wandb_run = wandb.init(
        project=CFG['project'],
        name=f"{CFG['run_name']}-fold{fold}-{run_id}",
        group=CFG['group'],
        job_type=f'fold_{fold}',
        config={**CFG, 'fold': fold, 'train_size': len(train_pairs), 'val_size': len(val_pairs)},
        dir=str(fold_out),
        anonymous='allow',
        reinit=True,
    )
    wandb.watch(model, log='gradients', log_freq=50)

    best_iou = -1.0
    best_dice = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, CFG['epochs'] + 1):
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats()
        train_m = train_one_epoch(model, train_loader, optimizer, scaler, total_loss, device)
        val_m, preview = validate(model, val_loader, total_loss, device, log_images=(epoch == 1 or epoch % 5 == 0 or epoch == CFG['epochs']))
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']
        gpu_stats = get_gpu_stats(device)

        row = {
            'fold': fold,
            'epoch': epoch,
            'lr': lr,
            'train_loss': train_m['loss'],
            'train_dice': train_m['dice'],
            'train_iou': train_m['iou'],
            'val_loss': val_m['loss'],
            'val_dice': val_m['dice'],
            'val_iou': val_m['iou'],
            **gpu_stats,
        }
        history.append(row)

        log_data = dict(row)
        if preview is not None:
            log_data['val_predictions'] = preview
        wandb.log(log_data, step=epoch)

        improved = val_m['iou'] > best_iou + CFG['early_stopping_min_delta']
        if improved:
            best_iou = val_m['iou']
            best_dice = val_m['dice']
            best_epoch = epoch
            epochs_without_improvement = 0
            ckpt = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'fold': fold,
                'best_iou': best_iou,
                'best_dice': best_dice,
                'config': CFG,
            }
            torch.save(ckpt, fold_out / 'best_unetpp.pt')
            wandb.save(str(fold_out / 'best_unetpp.pt'))
        else:
            epochs_without_improvement += 1

        pd.DataFrame(history).to_csv(fold_out / 'history.csv', index=False)
        print(
            f"Fold {fold} | Epoch {epoch:03d}/{CFG['epochs']} | "
            f"train loss {train_m['loss']:.4f} dice {train_m['dice']:.4f} iou {train_m['iou']:.4f} | "
            f"val loss {val_m['loss']:.4f} dice {val_m['dice']:.4f} iou {val_m['iou']:.4f} | "
            f"best {best_iou:.4f} @ {best_epoch} | "
            f"no_improve {epochs_without_improvement}/{CFG['early_stopping_patience']}"
        )

        if epochs_without_improvement >= CFG['early_stopping_patience']:
            print(f"Early stopping fold {fold}: no val_iou improvement for {CFG['early_stopping_patience']} epochs.")
            break

    wandb.finish()
    del model, optimizer, scheduler, scaler
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    return {
        'fold': fold,
        'best_epoch': best_epoch,
        'best_val_iou': best_iou,
        'best_val_dice': best_dice,
        'train_size': len(train_pairs),
        'val_size': len(val_pairs),
        'checkpoint': str(fold_out / 'best_unetpp.pt'),
    }


In [ ]:
fold_results = []
for fold in CFG['folds_to_run']:
    fold_results.append(run_fold(fold))

cv_summary = pd.DataFrame(fold_results)
cv_summary.to_csv(BASE_OUT_DIR / 'cv_summary.csv', index=False)
print(cv_summary)
print()
print('Mean metrics:')
print(cv_summary[['best_val_iou', 'best_val_dice']].agg(['mean', 'std']))


In [ ]:
# Optional: quick threshold sweep for each best checkpoint can be added after CV.
# Current training tracks validation metrics at threshold=CFG['threshold'].
